<a href="https://colab.research.google.com/github/mohamadatashfaraz4-netizen/master-thesis/blob/main/10_Empirische_Ergebnisse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install -q pandas openpyxl matplotlib

In [ ]:
from google.colab import files

# Ergebnisdateien aus vorherigen Kapiteln hochladen
uploaded = files.upload()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# Robustheitsexperiment aus Kapitel 9 laden
robustness_df = pd.read_excel("chapter09-robustness_experiment.xlsx")

# TabPFN-Ergebnisse aus Kapitel 7 laden
chapter07_results = pd.read_excel("chapter07_tabpfn_results_corrupted_data.xlsx")
uncertain_cases = pd.read_excel("chapter07_uncertain_cases_corrupted_data.xlsx")
tabpfn_cleaned = pd.read_excel("chapter07_tabpfn_supported_cleaned_data.xlsx")

print("Robustness results:", robustness_df.shape)
print("Chapter 7 results:", chapter07_results.shape)
print("Uncertain cases:", uncertain_cases.shape)
print("TabPFN-supported cleaned data:", tabpfn_cleaned.shape)

In [ ]:
# Saubere Referenz aus Robustheitsexperiment extrahieren
clean_reference = robustness_df[
    robustness_df["error_type"] == "clean_reference"
].copy()

# Ergebnisse bei 10 Prozent Fehleranteil extrahieren
level_10_results = robustness_df[
    (robustness_df["level"] == 0.10) &
    (robustness_df["error_type"] != "clean_reference")
].copy()

# Relevante Spalten auswählen
level_10_summary = level_10_results[
    [
        "error_type",
        "level",
        "accuracy",
        "roc_auc",
        "accuracy_drop",
        "roc_auc_drop",
        "mean_confidence",
        "uncertainty_f1"
    ]
]

level_10_summary

In [ ]:
# Anzahl der Review-Fälle aus Kapitel 7 berechnen
review_count = int(tabpfn_cleaned["tabpfn_needs_review"].sum())
total_rows = len(tabpfn_cleaned)
review_ratio = review_count / total_rows

# Kapitel-7-Zusammenfassung erstellen
chapter07_summary = pd.DataFrame({
    "metric": [
        "rows_in_tabpfn_supported_cleaned_data",
        "rows_marked_for_review",
        "review_ratio"
    ],
    "value": [
        total_rows,
        review_count,
        review_ratio
    ]
})

chapter07_summary

In [ ]:
# Zusammenfassung der unsicheren Fälle aus Kapitel 7
uncertain_summary = pd.DataFrame({
    "metric": [
        "uncertain_cases_total",
        "needs_review_true",
        "needs_review_false",
        "mean_max_probability",
        "min_max_probability",
        "max_max_probability"
    ],
    "value": [
        len(uncertain_cases),
        int(uncertain_cases["needs_review"].sum()),
        int((uncertain_cases["needs_review"] == False).sum()),
        float(uncertain_cases["max_probability"].mean()),
        float(uncertain_cases["max_probability"].min()),
        float(uncertain_cases["max_probability"].max()),
    ]
})

uncertain_summary

In [ ]:
# Zentrale Ergebnisübersicht erstellen
empirical_summary = pd.DataFrame({
    "analysis_part": [
        "Robustheitsexperiment",
        "Robustheitsexperiment",
        "Robustheitsexperiment",
        "Robustheitsexperiment",
        "TabPFN-supported Cleaning"
    ],
    "result_type": [
        "missing_values",
        "outliers",
        "wrong_labels",
        "feature_noise",
        "review_prioritization"
    ],
    "main_observation": [
        "Untersucht die Veränderung der Modellleistung bei künstlich erzeugten fehlenden Werten.",
        "Untersucht die Veränderung der Modellleistung bei künstlich erzeugten Ausreißern.",
        "Untersucht die Veränderung der Modellleistung bei fehlerhaften Labels.",
        "Untersucht die Veränderung der Modellleistung bei Feature Noise.",
        "TabPFN markiert unsichere Fälle für manuelle Prüfung und ergänzt klassische Bereinigung."
    ]
})

empirical_summary

In [ ]:
# Plot-Daten vorbereiten
plot_df = robustness_df[
    robustness_df["error_type"] != "clean_reference"
].copy()

label_map = {
    "missing_values": "Missing Values",
    "outliers": "Outliers",
    "wrong_labels": "Wrong Labels",
    "feature_noise": "Feature Noise"
}

plt.figure(figsize=(7.2, 4.4))

for error_type, part in plot_df.groupby("error_type"):
    plt.plot(
        part["level"] * 100,
        part["accuracy"],
        marker="o",
        label=label_map.get(error_type, error_type)
    )

plt.xlabel("Künstlich erzeugter Fehleranteil (%)")
plt.ylabel("Accuracy")
plt.title("Accuracy bei unterschiedlichen Datenfehlern")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

accuracy_plot = "chapter10_accuracy_by_error_type.png"
plt.savefig(accuracy_plot, dpi=180)
plt.show()

files.download(accuracy_plot)

In [ ]:
plt.figure(figsize=(7.2, 4.4))

for error_type, part in plot_df.groupby("error_type"):
    plt.plot(
        part["level"] * 100,
        part["roc_auc"],
        marker="o",
        label=label_map.get(error_type, error_type)
    )

plt.xlabel("Künstlich erzeugter Fehleranteil (%)")
plt.ylabel("ROC AUC")
plt.title("ROC AUC bei unterschiedlichen Datenfehlern")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

roc_auc_plot = "chapter10_roc_auc_by_error_type.png"
plt.savefig(roc_auc_plot, dpi=180)
plt.show()

files.download(roc_auc_plot)

In [ ]:
plt.figure(figsize=(7.2, 4.4))

for error_type, part in plot_df.groupby("error_type"):
    plt.plot(
        part["level"] * 100,
        part["mean_confidence"],
        marker="o",
        label=label_map.get(error_type, error_type)
    )

plt.xlabel("Künstlich erzeugter Fehleranteil (%)")
plt.ylabel("Mean Confidence")
plt.title("Mittlere Modellkonfidenz bei unterschiedlichen Datenfehlern")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

confidence_plot = "chapter10_mean_confidence_by_error_type.png"
plt.savefig(confidence_plot, dpi=180)
plt.show()

files.download(confidence_plot)

In [ ]:
# Dateinamen definieren
summary_file = "chapter10_empirical_summary.xlsx"
level10_file = "chapter10_level10_robustness_summary.xlsx"
chapter07_summary_file = "chapter10_tabpfn_supported_cleaning_summary.xlsx"
uncertain_summary_file = "chapter10_uncertain_cases_summary.xlsx"

# Dateien speichern
empirical_summary.to_excel(summary_file, index=False)
level_10_summary.to_excel(level10_file, index=False)
chapter07_summary.to_excel(chapter07_summary_file, index=False)
uncertain_summary.to_excel(uncertain_summary_file, index=False)

# Dateien herunterladen
files.download(summary_file)
files.download(level10_file)
files.download(chapter07_summary_file)
files.download(uncertain_summary_file)

print("Saved files:")
print(summary_file)
print(level10_file)
print(chapter07_summary_file)
print(uncertain_summary_file)